In [2]:
# %% [code]
!pip install -q stardist csbdeep zarr geff networkx scipy scikit-image numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 24.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.7/363.7 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 96.9 MB/s eta 0:00:00:00:0100:01


In [3]:
# imports
import os
import numpy as np
import zarr
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.ndimage as ndi
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from skimage.registration import phase_cross_correlation
from stardist.models import StarDist3D
from csbdeep.utils import normalize

# 1. Setup GPU Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Load StarDist3D (Downloads weights on first run)
print("Loading StarDist3D...")
stardist_model = StarDist3D.from_pretrained('3D_demo')
print("StarDist3D loaded successfully!")

Using device: cuda
Loading StarDist3D...
Found model '3D_demo' for 'StarDist3D'.
5749101/5749101 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


I0000 00:00:1789445009.362198      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789445009.365090      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.707933, nms_thresh=0.3.
StarDist3D loaded successfully!


In [ ]:
# features
def preprocess_3d_volume(vol, alpha=1.2, sigma=(0.8, 1.5, 1.5)):
    """Applies 3D edge detection and unsharp masking to sharpen cell boundaries."""
    vol_float = vol.astype(np.float32)
    blurred = ndi.gaussian_filter(vol_float, sigma=sigma)
    edges = vol_float - blurred
    sharpened = vol_float + alpha * edges
    return np.clip(sharpened, 0, None)

def get_motion_vector(centroid, label_id, labels_T, frame_T, frame_T1, r=10):
    """Calculates 3D optical flow/shift for a specific cell using masked cross-correlation."""
    z, y, x = int(centroid[0]), int(centroid[1]), int(centroid[2])
    Z_max, Y_max, X_max = frame_T.shape

    z_min, z_max = max(0, z-r), min(Z_max, z+r)
    y_min, y_max = max(0, y-r), min(Y_max, y+r)
    x_min, x_max = max(0, x-r), min(X_max, x+r)

    crop_T = frame_T[z_min:z_max, y_min:y_max, x_min:x_max].copy()
    mask_crop = labels_T[z_min:z_max, y_min:y_max, x_min:x_max]
    
    # Zero out background / neighbor cells
    crop_T[mask_crop != label_id] = 0

    # Avoid warnings if the cell mask is entirely empty (edge of frame)
    if np.all(crop_T == 0):
        return np.array([0.0, 0.0, 0.0])

    crop_T1 = frame_T1[z_min:z_max, y_min:y_max, x_min:x_max]

    if crop_T.shape == crop_T1.shape and 0 not in crop_T.shape:
        shift, _, _ = phase_cross_correlation(crop_T, crop_T1, upsample_factor=10)
        return shift
    else:
        return np.array([0.0, 0.0, 0.0])

In [ ]:
# deep network
# =======================================
# 1. Appearance Embedding Network
# =======================================
class CellReID3D(nn.Module):
    def __init__(self, emb_dim=64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(1, 16, kernel_size=3, padding=1), nn.BatchNorm3d(16), nn.ReLU(inplace=True), nn.MaxPool3d(2),
            nn.Conv3d(16, 32, kernel_size=3, padding=1), nn.BatchNorm3d(32), nn.ReLU(inplace=True), nn.AdaptiveAvgPool3d((2, 2, 2))
        )
        self.fc = nn.Linear(32 * 2 * 2 * 2, emb_dim)

    def forward(self, x):
        feat = self.conv(x)
        feat = feat.view(feat.size(0), -1)
        return F.normalize(self.fc(feat), p=2, dim=1) # L2 normalize for Cosine Distance

reid_model = CellReID3D(emb_dim=64).to(device)
reid_model.eval()

def extract_cell_embeddings(volume, centroids, crop_size=16):
    if len(centroids) == 0: return np.empty((0, 64))
    r = crop_size // 2
    vol_padded = np.pad(volume, r, mode='reflect')
    crops = []
    
    for c in centroids:
        cz, cy, cx = int(round(c[0])) + r, int(round(c[1])) + r, int(round(c[2])) + r
        crops.append(vol_padded[cz-r:cz+r, cy-r:cy+r, cx-r:cx+r])
        
    batch = torch.tensor(np.stack(crops), dtype=torch.float32).unsqueeze(1).to(device)
    batch = (batch - batch.mean(dim=(2,3,4), keepdim=True)) / (batch.std(dim=(2,3,4), keepdim=True) + 1e-6)
    
    with torch.no_grad():
        embeddings = reid_model(batch)
    return embeddings.cpu().numpy()

# =======================================
# 2. DeepSORT Tracker
# =======================================
class DeepTrack3D:
    def __init__(self, track_id, centroid, label_id, embedding):
        self.track_id = track_id
        self.current_centroid = centroid
        self.current_label = label_id
        self.motion_vector = np.array([0.0, 0.0, 0.0])
        self.features = [embedding] # Gallery
        self.time_since_update = 0

    def update(self, centroid, label_id, embedding):
        self.current_centroid = centroid
        self.current_label = label_id
        self.features.append(embedding)
        if len(self.features) > 5: self.features.pop(0)
        self.time_since_update = 0

class DeepSORT3D:
    def __init__(self, max_age=3, max_dist=12.0, max_cosine_dist=0.3, lambda_weight=0.6):
        self.tracks = []
        self.next_track_id = 1
        self.max_age, self.max_dist, self.max_cosine_dist, self.lambda_weight = max_age, max_dist, max_cosine_dist, lambda_weight
        self.graph = nx.DiGraph()

    def update(self, frame_T, frame_T1, labels_T, centroids_T1, labels_T1, embeddings_T1, timepoint):
        # 1. Predict motion
        for track in self.tracks:
            track.motion_vector = get_motion_vector(track.current_centroid, track.current_label, labels_T, frame_T, frame_T1)

        unmatched_dets = list(range(len(centroids_T1)))
        matched_pairs = []

        # 2. Matching Cascade (Match newest tracks first)
        for age in range(self.max_age + 1):
            if len(unmatched_dets) == 0: break
            active_tracks = [t for t in self.tracks if t.time_since_update == age]
            if len(active_tracks) == 0: continue

            curr_dets = [centroids_T1[i] for i in unmatched_dets]
            curr_embeds = np.array([embeddings_T1[i] for i in unmatched_dets])

            # Motion Cost
            pred_positions = [t.current_centroid + t.motion_vector for t in active_tracks]
            motion_cost = cdist(np.array(pred_positions), curr_dets, metric='euclidean')

            # Appearance Cost
            app_cost = np.zeros_like(motion_cost)
            for idx, track in enumerate(active_tracks):
                track_feats = np.array(track.features)
                dists = 1.0 - np.dot(curr_embeds, track_feats.T)
                app_cost[idx, :] = np.min(dists, axis=1)

            # Fused Cost & Gates
            fused_cost = self.lambda_weight * (motion_cost / self.max_dist) + (1.0 - self.lambda_weight) * app_cost
            fused_cost[motion_cost > self.max_dist] = 1e5
            fused_cost[app_cost > self.max_cosine_dist] = 1e5

            # Match
            row_ind, col_ind = linear_sum_assignment(fused_cost)
            for r, c in zip(row_ind, col_ind):
                if fused_cost[r, c] < 1e4:
                    matched_pairs.append((active_tracks[r], unmatched_dets[c]))

            # Prune matched from unmatched pool
            matched_det_idx = [det_idx for (_, det_idx) in matched_pairs]
            unmatched_dets = [i for i in unmatched_dets if i not in matched_det_idx]

        # Helper for label extraction
        def get_label(c, labels_img, d_idx):
            cz = int(np.clip(round(c[0]), 0, labels_img.shape[0]-1))
            cy = int(np.clip(round(c[1]), 0, labels_img.shape[1]-1))
            cx = int(np.clip(round(c[2]), 0, labels_img.shape[2]-1))
            lbl = labels_img[cz, cy, cx]
            return lbl if lbl != 0 else (d_idx + 1)

        # 3. Update Tracks
        for track, det_idx in matched_pairs:
            new_c = centroids_T1[det_idx]
            
            # Record edge in graph
            self.graph.add_node(f"T{timepoint}_ID{track.track_id}", t=timepoint, z=new_c[0], y=new_c[1], x=new_c[2])
            self.graph.add_edge(f"T{timepoint-1}_ID{track.track_id}", f"T{timepoint}_ID{track.track_id}")
            
            track.update(new_c, get_label(new_c, labels_T1, det_idx), embeddings_T1[det_idx])

        # 4. Births & Deaths
        for det_idx in unmatched_dets:
            new_c = centroids_T1[det_idx]
            self.graph.add_node(f"T{timepoint}_ID{self.next_track_id}", t=timepoint, z=new_c[0], y=new_c[1], x=new_c[2])
            self.tracks.append(DeepTrack3D(self.next_track_id, new_c, get_label(new_c, labels_T1, det_idx), embeddings_T1[det_idx]))
            self.next_track_id += 1

        matched_tracks = [t for (t, _) in matched_pairs]
        for t in self.tracks:
            if t not in matched_tracks: t.time_since_update += 1
        self.tracks = [t for t in self.tracks if t.time_since_update <= self.max_age]

In [ ]:
# run
# 1. Load Data
zarr_path = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train/44b6_0113de3b.zarr"
print(f"Loading data from {zarr_path}...")
z = zarr.open(zarr_path, mode="r")
arr = z['0'] 
MAX_FRAMES = 20

deepsort_tracker = DeepSORT3D(max_age=3, max_dist=12.0, max_cosine_dist=0.3)
print(f"\n--- Starting 3D DeepSORT Tracking ({MAX_FRAMES} frames) ---")

# --- FRAME 0 INIT ---
frame_0_sharp = preprocess_3d_volume(np.asarray(arr[0]))
labels_prev, details_prev = stardist_model.predict_instances(normalize(frame_0_sharp, 1, 99.8, axis=(0,1,2)))
centroids_prev = details_prev['points']
embeddings_prev = extract_cell_embeddings(frame_0_sharp, centroids_prev)

for i, c in enumerate(centroids_prev):
    cz, cy, cx = int(np.clip(round(c[0]), 0, labels_prev.shape[0]-1)), int(np.clip(round(c[1]), 0, labels_prev.shape[1]-1)), int(np.clip(round(c[2]), 0, labels_prev.shape[2]-1))
    lbl = labels_prev[cz, cy, cx] if labels_prev[cz, cy, cx] != 0 else (i + 1)
    
    deepsort_tracker.tracks.append(DeepTrack3D(deepsort_tracker.next_track_id, c, lbl, embeddings_prev[i]))
    deepsort_tracker.graph.add_node(f"T0_ID{deepsort_tracker.next_track_id}", t=0, z=c[0], y=c[1], x=c[2])
    deepsort_tracker.next_track_id += 1
    
print(f"Frame T=0: Spawned {len(centroids_prev)} tracks.")

# --- TIME LOOP ---
frame_prev_sharp = frame_0_sharp

for t in range(1, MAX_FRAMES):
    print(f"Processing Frame T={t}...")
    
    frame_curr_sharp = preprocess_3d_volume(np.asarray(arr[t]))
    labels_curr, details_curr = stardist_model.predict_instances(normalize(frame_curr_sharp, 1, 99.8, axis=(0,1,2)))
    centroids_curr = details_curr['points']
    embeddings_curr = extract_cell_embeddings(frame_curr_sharp, centroids_curr)
    
    deepsort_tracker.update(
        frame_prev_sharp, frame_curr_sharp, labels_prev, 
        centroids_curr, labels_curr, embeddings_curr, t
    )
    
    labels_prev = labels_curr
    frame_prev_sharp = frame_curr_sharp

print("\n==========================================")
print("3D DeepSORT Tracking Complete!")
print(f"Total Detections (Nodes): {deepsort_tracker.graph.number_of_nodes()}")
print(f"Total Connections (Edges): {deepsort_tracker.graph.number_of_edges()}")
print("==========================================")

In [ ]:
# eval not correct, just timepass
from geff import read
from scipy.spatial import KDTree

print("Loading Ground Truth .geff graph...")
geff_path = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train/44b6_0113de3b.geff"
gt_graph, metadata = read(geff_path, backend="networkx")

# 1. Extract Ground Truth nodes for t < MAX_FRAMES
gt_nodes_by_time = {t: [] for t in range(MAX_FRAMES)}
for node, data in gt_graph.nodes(data=True):
    t = data.get('t', data.get('time', None))
    if t is not None and t < MAX_FRAMES:
        gt_nodes_by_time[t].append((node, np.array([data['z'], data['y'], data['x']])))

# 2. Extract DeepSORT predicted nodes
pred_nodes_by_time = {t: [] for t in range(MAX_FRAMES)}
for node, data in deepsort_tracker.graph.nodes(data=True):
    t = data.get('t')
    if t is not None and t < MAX_FRAMES and 'z' in data:
        pred_nodes_by_time[t].append((node, np.array([data['z'], data['y'], data['x']])))

# 3. Match GT cells to predictions (16 voxel tolerance for border cells)
gt_to_pred_map = {}
MATCH_TOLERANCE = 16.0 

for t in range(MAX_FRAMES):
    if len(gt_nodes_by_time[t]) == 0 or len(pred_nodes_by_time[t]) == 0:
        continue
    pred_ids, pred_coords = zip(*pred_nodes_by_time[t])
    tree = KDTree(pred_coords)
    
    for gt_id, gt_coord in gt_nodes_by_time[t]:
        dist, idx = tree.query(gt_coord)
        if dist <= MATCH_TOLERANCE:
            gt_to_pred_map[gt_id] = pred_ids[idx]

print(f"Matched {len(gt_to_pred_map)} ground truth cell observations to DeepSORT detections.")

# 4. Check edge agreement (Did DeepSORT draw the same links?)
correct_edges = 0
total_gt_edges = 0

for u, v in gt_graph.edges():
    u_time = gt_graph.nodes[u].get('t', gt_graph.nodes[u].get('time', None))
    v_time = gt_graph.nodes[v].get('t', gt_graph.nodes[v].get('time', None))
    
    if u_time is not None and v_time is not None and max(u_time, v_time) < MAX_FRAMES:
        total_gt_edges += 1
        if u in gt_to_pred_map and v in gt_to_pred_map:
            pred_u = gt_to_pred_map[u]
            pred_v = gt_to_pred_map[v]
            if deepsort_tracker.graph.has_edge(pred_u, pred_v):
                correct_edges += 1

print("\n==========================================")
print("      3D DeepSORT BENCHMARK REPORT        ")
print("==========================================")
print(f"Total Ground Truth Edges (t < {MAX_FRAMES}): {total_gt_edges}")
print(f"Correctly Tracked Edges: {correct_edges}")

if total_gt_edges > 0:
    recall = (correct_edges / total_gt_edges) * 100
    print(f"Tracking Recall: {recall:.2f}%")
print("==========================================")

In [ ]:
# view
import matplotlib.pyplot as plt

# 1. Collect points per frame
gt_coords_by_t = {t: [] for t in range(MAX_FRAMES)}
for node, data in gt_graph.nodes(data=True):
    t = data.get('t', data.get('time', None))
    if t is not None and t < MAX_FRAMES:
        gt_coords_by_t[t].append((data['x'], data['y']))

pred_coords_by_t = {t: [] for t in range(MAX_FRAMES)}
for node, data in deepsort_tracker.graph.nodes(data=True):
    t = data.get('t')
    if t is not None and t < MAX_FRAMES and 'x' in data:
        pred_coords_by_t[t].append((data['x'], data['y']))

# 2. Render 5-Frame Gallery
fig, axes = plt.subplots(1, MAX_FRAMES, figsize=(4 * MAX_FRAMES, 5))
if MAX_FRAMES == 1: axes = [axes]

for t in range(MAX_FRAMES):
    vol = np.asarray(arr[t])
    mip_xy = vol.max(axis=0)
    
    lo, hi = np.percentile(mip_xy, [1, 99.5])
    mip_norm = np.clip((mip_xy - lo) / (hi - lo + 1e-8), 0, 1)
    
    ax = axes[t]
    ax.imshow(mip_norm, cmap='gray')
    ax.set_title(f"Timepoint T = {t}", fontsize=12, fontweight='bold')
    ax.axis('off')
    
    # Plot Detections
    if len(pred_coords_by_t[t]) > 0:
        px, py = zip(*pred_coords_by_t[t])
        ax.scatter(px, py, c='cyan', s=14, alpha=0.7, label='DeepSORT' if t == 0 else "")
        
    # Plot Ground Truth
    if len(gt_coords_by_t[t]) > 0:
        gx, gy = zip(*gt_coords_by_t[t])
        ax.scatter(gx, gy, c='red', s=100, marker='o', edgecolors='white', linewidth=1.5, label='Ground Truth' if t == 0 else "")

axes[0].legend(loc='upper left', framealpha=0.8)
plt.tight_layout()
plt.savefig("/kaggle/working/deepsort_gallery.png", dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# trajectory
# 1. Reconstruct continuous tracks from the graph
track_paths = {}
for node, data in deepsort_tracker.graph.nodes(data=True):
    if '_ID' in node and 't' in data and 'x' in data:
        track_id = node.split('_ID')[1]
        if track_id not in track_paths:
            track_paths[track_id] = []
        track_paths[track_id].append((data['t'], data['z'], data['y'], data['x']))

# Sort points chronologically
for t_id in track_paths:
    track_paths[t_id].sort(key=lambda item: item[0])

# 2. Render Trajectory Plot
fig, ax = plt.subplots(figsize=(10, 10))

# Background: Last frame MIP
vol_last = np.asarray(arr[MAX_FRAMES - 1])
mip_last = vol_last.max(axis=0)
lo, hi = np.percentile(mip_last, [1, 99.5])
ax.imshow(np.clip((mip_last - lo) / (hi - lo + 1e-8), 0, 1), cmap='gray', alpha=0.8)
ax.set_title("Cell Migration Trajectories (DeepSORT 3D)", fontsize=14, fontweight='bold')

colors = plt.cm.rainbow(np.linspace(0, 1, len(track_paths)))
plotted_tracks = 0

for i, (t_id, path) in enumerate(track_paths.items()):
    if len(path) > 1: # Only plot lineages that were successfully tracked
        plotted_tracks += 1
        xs = [p[3] for p in path]
        ys = [p[2] for p in path]
        ax.plot(xs, ys, color=colors[i], linewidth=2.0, alpha=0.8)
        ax.scatter(xs[-1], ys[-1], color=colors[i], s=25, edgecolors='white', linewidth=0.5)

# Overlay Ground Truth Track
gt_track_coords = []
for t in range(MAX_FRAMES):
    if len(gt_coords_by_t[t]) > 0:
        gt_track_coords.append(gt_coords_by_t[t][0])

if len(gt_track_coords) > 1:
    gt_xs, gt_ys = zip(*gt_track_coords)
    ax.plot(gt_xs, gt_ys, color='red', linewidth=3.5, label='Ground Truth Track')
    ax.scatter(gt_xs, gt_ys, color='yellow', s=70, edgecolors='red', linewidth=1.5, zorder=5)

ax.legend(loc='upper left', fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.savefig("/kaggle/working/deepsort_trajectories.png", dpi=200, bbox_inches='tight')
plt.show()

print(f"Rendered trajectories for {plotted_tracks} continuous cell tracks!")

In [ ]:
# metrics
track_lengths = [len(path) for path in track_paths.values()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# 1. Track Length Histogram
ax1.hist(track_lengths, bins=range(1, MAX_FRAMES + 2), align='left', rwidth=0.8, color='teal', edgecolor='black')
ax1.set_title("Track Lifetime Distribution", fontsize=12, fontweight='bold')
ax1.set_xlabel("Track Length (Frames)")
ax1.set_ylabel("Number of Cells")
ax1.grid(axis='y', alpha=0.3)

# 2. Detections per Frame
counts = [len(pred_coords_by_t[t]) for t in range(MAX_FRAMES)]
ax2.plot(range(MAX_FRAMES), counts, marker='o', color='crimson', linewidth=2)
ax2.set_title("Cell Detections per Timepoint", fontsize=12, fontweight='bold')
ax2.set_xlabel("Timepoint T")
ax2.set_ylabel("Active Detections")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# video
import cv2
import numpy as np
import imageio
from IPython.display import Image, display

frames = []

for t in range(MAX_FRAMES):
    # 1. 2D Maximum Intensity Projection
    vol = np.asarray(arr[t])
    mip = vol.max(axis=0)  # Shape (256, 256)
    
    # 2. Normalize to 0-255 uint8
    lo, hi = np.percentile(mip, [1, 99.5])
    mip_norm = np.clip((mip - lo) / (hi - lo + 1e-8), 0, 1)
    mip_uint8 = (mip_norm * 255).astype(np.uint8)
    
    # 3. Convert grayscale to 3-channel RGB image
    img_rgb = cv2.cvtColor(mip_uint8, cv2.COLOR_GRAY2RGB)
    
    # 4. Draw Cyan Dots for DeepSORT Detections (R=0, G=255, B=255)
    if t in pred_coords_by_t:
        for x, y in pred_coords_by_t[t]:
            cv2.circle(img_rgb, (int(round(x)), int(round(y))), radius=2, color=(0, 255, 255), thickness=-1)
            
    # 5. Draw Large Red Ring for Ground Truth (R=255, G=0, B=0)
    if t in gt_coords_by_t:
        for x, y in gt_coords_by_t[t]:
            cv2.circle(img_rgb, (int(round(x)), int(round(y))), radius=6, color=(255, 0, 0), thickness=2)
            
    frames.append(img_rgb)

# Save as a lightweight GIF (~150 KB) and display directly
gif_path = "/kaggle/working/tracker_preview.gif"
imageio.mimsave(gif_path, frames, duration=400, loop=0)

print("Displaying lightweight GIF...")
display(Image(filename=gif_path))

In [ ]:
# 3d
import plotly.graph_objects as go
import numpy as np

# Physical voxel scale from the competition metadata:
# Z resolution is coarser (1.625 µm) compared to X/Y (0.40625 µm)
Z_SCALE = 1.625 / 0.40625  # ~4.0x stretch along Z so cells aren't squashed

fig = go.Figure()

# 1. Reconstruct 3D tracks from deepsort_tracker.graph
track_paths = {}
for node, data in deepsort_tracker.graph.nodes(data=True):
    if '_ID' in node and 't' in data and 'x' in data:
        t_id = node.split('_ID')[1]
        if t_id not in track_paths:
            track_paths[t_id] = []
        track_paths[t_id].append((data['t'], data['z'], data['y'], data['x']))

for t_id in track_paths:
    track_paths[t_id].sort(key=lambda item: item[0])

# 2. Draw 3D Trajectory Lines for DeepSORT predictions
for t_id, path in track_paths.items():
    if len(path) > 1: # Only plot cells tracked for more than 1 frame
        ts = [p[0] for p in path]
        zs = [p[1] * Z_SCALE for p in path]
        ys = [p[2] for p in path]
        xs = [p[3] for p in path]
        
        # Draw the 3D line
        fig.add_trace(go.Scatter3d(
            x=xs, y=ys, z=zs,
            mode='lines+markers',
            line=dict(width=4),
            marker=dict(size=3),
            hoverinfo='text',
            text=[f"ID: {t_id} | T={t} | z={z/Z_SCALE:.1f}" for t, z in zip(ts, zs)],
            showlegend=False
        ))

# 3. Draw the Ground Truth 3D Trajectory (Thick Red/Yellow in 3D)
gt_path = []
for t in range(MAX_FRAMES):
    if len(gt_nodes_by_time[t]) > 0:
        # Take the first GT cell
        gt_node, coord = gt_nodes_by_time[t][0]
        gt_path.append((t, coord[0], coord[1], coord[2]))

if len(gt_path) > 1:
    gt_ts = [p[0] for p in gt_path]
    gt_zs = [p[1] * Z_SCALE for p in gt_path]
    gt_ys = [p[2] for p in gt_path]
    gt_xs = [p[3] for p in gt_path]
    
    fig.add_trace(go.Scatter3d(
        x=gt_xs, y=gt_ys, z=gt_zs,
        mode='lines+markers',
        line=dict(color='red', width=10),
        marker=dict(color='yellow', size=8, symbol='diamond'),
        name='Ground Truth 3D Track'
    ))

# 4. Format 3D Scene
fig.update_layout(
    title="<b>Interactive 3D Cell Lineage Tracks (Rotate with your mouse!)</b>",
    scene=dict(
        xaxis_title='X (voxels)',
        yaxis_title='Y (voxels)',
        zaxis_title='Z (scaled µm)',
        aspectmode='data', # Keeps realistic 3D physical proportions
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.2) # Default viewing angle
        )
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    height=700
)

fig.show()

In [ ]:
# %% [code]
# --- Cell 2: GPU-accelerated checkpointed sweep ---
import os
import glob
import time
import random
import json
import numpy as np
import pandas as pd
import zarr
import cupy as cp
import cupyx.scipy.ndimage as cundi
from geff import read
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

TIME_BUDGET_SEC = 5 * 3600
CHECKPOINT_PATH = "/kaggle/working/dog_sweep_checkpoint.csv"
SEED = 42
N_SAMPLES = 300  # GPU is ~30-40x faster — push this way up vs the CPU plan

MAX_DIST_UM = 7.0

RADIUS_GRID = [2.0, 2.5, 3.0, 3.5, 4.0, 5.0]
THRESH_GRID = [0.02, 0.03, 0.05, 0.07, 0.10, 0.15]
MIN_DIST_GRID = [2, 3, 4]

base = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train"
if not os.path.isdir(base):
    base = "/kaggle/input/biohub-cell-tracking-during-development/train"

random.seed(SEED)
zarr_paths = sorted(glob.glob(os.path.join(base, "*.zarr")))
all_names = [os.path.basename(p).replace(".zarr", "") for p in zarr_paths]
all_names = [n for n in all_names if os.path.exists(os.path.join(base, n + ".geff"))]
chosen = random.sample(all_names, min(N_SAMPLES, len(all_names)))
print(f"Sweeping over {len(chosen)} samples")

if os.path.exists(CHECKPOINT_PATH):
    done_df = pd.read_csv(CHECKPOINT_PATH)
    done_keys = set(zip(done_df.sample_name, done_df.radius_um, done_df.thresh_rel, done_df.min_dist))
    print(f"Resuming — {len(done_df)} combos already completed.")
else:
    done_df = pd.DataFrame()
    done_keys = set()

def dog_sigmas(radius_um, z_scale, xy_scale):
    s_xy = radius_um / (1.414 * xy_scale)
    s_z = radius_um / (1.414 * z_scale)
    small = (s_z, s_xy, s_xy)
    return small, tuple(s * 1.6 for s in small)

def peak_local_max_gpu(vol_gpu, min_distance, threshold_rel):
    size = 2 * min_distance + 1
    local_max = cundi.maximum_filter(vol_gpu, size=size, mode='constant', cval=-cp.inf)
    thresh_val = threshold_rel * vol_gpu.max()
    mask = (vol_gpu == local_max) & (vol_gpu > thresh_val)
    return cp.argwhere(mask)

def load_sample(name):
    z_root = zarr.open(os.path.join(base, name + ".zarr"), mode="r")
    arr = z_root['0']
    stack = np.asarray(arr[:]).astype(np.float32)

    gt_graph, metadata = read(os.path.join(base, name + ".geff"), backend="networkx")
    T_true = metadata.extra["estimated_number_of_nodes"]
    scale_map = {ax.name: ax.scale for ax in metadata.axes if ax.type == "space"}
    SCALE = np.array([scale_map["z"], scale_map["y"], scale_map["x"]])

    gt_by_t = {}
    for node, data in gt_graph.nodes(data=True):
        t = data.get('t', data.get('time'))
        if t is None:
            continue
        coord = np.array([data['z'], data['y'], data['x']]) * SCALE
        gt_by_t.setdefault(t, []).append((node, coord))

    return stack, scale_map, SCALE, gt_by_t, T_true

results_buffer = []
start_time = time.time()
combos_done = 0
combos_skipped = 0
stop_early = False

for name in chosen:
    if stop_early:
        break
    print(f"\n=== Sample: {name} ===")
    stack, scale_map, SCALE, gt_by_t, T_true = load_sample(name)
    n_frames = stack.shape[0]
    stack_gpu = cp.asarray(stack, dtype=cp.float32)  # whole sample resident on GPU

    for radius in RADIUS_GRID:
        if stop_early:
            break
        small, large = dog_sigmas(radius, scale_map["z"], scale_map["y"])

        # DoG computed once per (sample, radius), kept on GPU, reused across thresh/min_dist
        t0 = time.time()
        dog_gpu_stack = cp.empty_like(stack_gpu)
        for t in range(n_frames):
            s = cundi.gaussian_filter(stack_gpu[t], sigma=small)
            l = cundi.gaussian_filter(stack_gpu[t], sigma=large)
            dog_gpu_stack[t] = s - l
        cp.cuda.Stream.null.synchronize()
        filter_elapsed = time.time() - t0

        for min_dist in MIN_DIST_GRID:
            for thresh in THRESH_GRID:
                key = (name, radius, thresh, min_dist)
                if key in done_keys:
                    combos_skipped += 1
                    continue

                if time.time() - start_time > TIME_BUDGET_SEC:
                    print("Time budget reached — stopping and saving checkpoint.")
                    stop_early = True
                    break

                pred_by_t = {}
                T_pred = 0
                for t in range(n_frames):
                    coords_gpu = peak_local_max_gpu(dog_gpu_stack[t], min_dist, thresh)
                    centroids = cp.asnumpy(coords_gpu)
                    if len(centroids) == 0:
                        continue
                    pred_by_t[t] = list(enumerate(centroids * SCALE))
                    T_pred += len(centroids)

                matched_gt, total_gt = 0, 0
                for t in gt_by_t:
                    total_gt += len(gt_by_t[t])
                    if t not in pred_by_t:
                        continue
                    gt_ids, gt_coords = zip(*gt_by_t[t])
                    pred_ids, pred_coords = zip(*pred_by_t[t])
                    cost = cdist(np.array(gt_coords), np.array(pred_coords))
                    row_ind, col_ind = linear_sum_assignment(cost)
                    for r, c in zip(row_ind, col_ind):
                        if cost[r, c] <= MAX_DIST_UM:
                            matched_gt += 1

                recall = matched_gt / total_gt if total_gt else 0.0
                over = (T_pred - T_true) / T_true if T_true else 0.0
                penalty_mult = max(0.0, 1 - 0.1 * over)
                combined_score = recall * penalty_mult

                results_buffer.append({
                    "sample_name": name, "radius_um": radius, "thresh_rel": thresh,
                    "min_dist": min_dist, "matched_gt": matched_gt, "total_gt": total_gt,
                    "T_pred": T_pred, "T_true": T_true, "recall": recall,
                    "over_pct": over * 100, "penalty_mult": penalty_mult,
                    "combined_score": combined_score
                })
                combos_done += 1

                if len(results_buffer) >= 50:
                    pd.concat([done_df, pd.DataFrame(results_buffer)], ignore_index=True) \
                      .to_csv(CHECKPOINT_PATH, index=False)
                    done_df = pd.read_csv(CHECKPOINT_PATH)
                    results_buffer = []

            if stop_early:
                break
        elapsed_h = (time.time() - start_time) / 3600
        print(f"  radius={radius}: filter {filter_elapsed:.1f}s | "
              f"elapsed {elapsed_h:.2f}h/{TIME_BUDGET_SEC/3600:.1f}h | "
              f"combos: {combos_done} done, {combos_skipped} resumed")

    del stack_gpu, dog_gpu_stack
    cp.get_default_memory_pool().free_all_blocks()

if results_buffer:
    pd.concat([done_df, pd.DataFrame(results_buffer)], ignore_index=True) \
      .to_csv(CHECKPOINT_PATH, index=False)

print(f"\nDone. {combos_done} new combos this run. Results at {CHECKPOINT_PATH}")

Sweeping over 199 samples

=== Sample: 6bba_b693381b ===
  radius=2.0: filter 9.4s | elapsed 0.01h/5.0h | combos: 18 done, 0 resumed
  radius=2.5: filter 9.0s | elapsed 0.01h/5.0h | combos: 36 done, 0 resumed
  radius=3.0: filter 9.0s | elapsed 0.02h/5.0h | combos: 54 done, 0 resumed
  radius=3.5: filter 7.8s | elapsed 0.02h/5.0h | combos: 72 done, 0 resumed
  radius=4.0: filter 5.5s | elapsed 0.02h/5.0h | combos: 90 done, 0 resumed
  radius=5.0: filter 9.6s | elapsed 0.03h/5.0h | combos: 108 done, 0 resumed

=== Sample: 44b6_74d0c52e ===
  radius=2.0: filter 1.0s | elapsed 0.03h/5.0h | combos: 126 done, 0 resumed
  radius=2.5: filter 1.0s | elapsed 0.03h/5.0h | combos: 144 done, 0 resumed
  radius=3.0: filter 1.2s | elapsed 0.03h/5.0h | combos: 162 done, 0 resumed
  radius=3.5: filter 1.5s | elapsed 0.04h/5.0h | combos: 180 done, 0 resumed
  radius=4.0: filter 1.7s | elapsed 0.04h/5.0h | combos: 198 done, 0 resumed
  radius=5.0: filter 2.1s | elapsed 0.04h/5.0h | combos: 216 done, 0 r

In [5]:
# %% [code]
# --- Cell 3: Analyze results (run any time, even mid-sweep) ---
df = pd.read_csv(CHECKPOINT_PATH)

agg = df.groupby(["radius_um", "thresh_rel", "min_dist"]).agg(
    total_matched=("matched_gt", "sum"),
    total_gt=("total_gt", "sum"),
    total_pred=("T_pred", "sum"),
    total_true=("T_true", "sum"),
    n_samples=("sample_name", "nunique")
).reset_index()

agg["recall"] = agg.total_matched / agg.total_gt
agg["over_pct"] = (agg.total_pred - agg.total_true) / agg.total_true * 100
agg["penalty_mult"] = (1 - 0.1 * (agg.total_pred - agg.total_true) / agg.total_true).clip(lower=0)
agg["combined_score"] = agg.recall * agg.penalty_mult

top10 = agg.sort_values("combined_score", ascending=False).head(10)
print(f"Configs tested: {len(agg)} | Total combo-runs logged: {len(df)}")
print(top10[["radius_um","thresh_rel","min_dist","recall","over_pct","penalty_mult","combined_score","n_samples"]]
      .to_string(index=False))

Configs tested: 108 | Total combo-runs logged: 21492
 radius_um  thresh_rel  min_dist   recall  over_pct  penalty_mult  combined_score  n_samples
       2.5        0.03         2 0.950097 18.721674      0.981278        0.932309        199
       2.5        0.02         2 0.952790 21.864326      0.978136        0.931957        199
       2.5        0.05         2 0.944666 15.030316      0.984970        0.930468        199
       2.5        0.03         3 0.938635 11.654674      0.988345        0.927696        199
       2.5        0.02         3 0.941126 14.380935      0.985619        0.927591        199
       2.5        0.07         2 0.938118 11.842860      0.988157        0.927008        199
       2.0        0.03         2 0.966831 41.801822      0.958198        0.926416        199
       2.0        0.03         3 0.955527 30.651558      0.969348        0.926239        199
       3.0        0.02         2 0.929124  3.485988      0.996514        0.925885        199
       3.0       